# `fig_study_area` — Okavango Delta Study Region

**Purpose:** Publication-quality study area figure.

**Panels:**
- **(a)** Main map — Africa locator inset + zoomed Okavango Delta with polygon boundary, river network, key GRDC gauge stations  
- **(b)** DSWE hydroperiod seasonal stack — monthly-mean inundated area by calendar month (Landsat 1987–, Sentinel-2 2019–)  
- **(c)** Ground-truth photo insets at key field sites

**Output:** `../figures/study_area/fig_study_area.{png,pdf}`

## 1 · Import Libraries and Configure Paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
from matplotlib.offsetbox import AnchoredOffsetbox, AuxTransformBox
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    warnings.warn("cartopy not found – falling back to plain matplotlib/contextily map")

try:
    import contextily as cx
    HAS_CONTEXTILY = True
except ImportError:
    HAS_CONTEXTILY = False

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT        = Path("..").resolve()           # project root
DATA        = ROOT / "data"
FIG_OUT     = ROOT / "figures" / "study_area"
FIG_OUT.mkdir(parents=True, exist_ok=True)

DELTA_SHP   = DATA / "regions" / "Delta_UCB_WGS84" / "Delta_UCB_WGS84.shp"
REGIONS_PKG = DATA / "regions" / "okavango_regions.gpkg"
HIGHLANDS   = DATA / "regions" / "okavango_highlands.geojson"
GRDC_META   = DATA / "grdc_metadata_compiled.csv"

LANDSAT_DSWE   = DATA / "monthly_landsat_dswe.csv"
SENTINEL_DSWE  = DATA / "monthly_sentinel2_dswe.csv"
ANNUAL_FLOOD   = DATA / "annual_flood_area_km2.csv"
MOHEMBO_DAILY  = DATA / "mohembo_1357100_Q_daily.csv"

# ── Ground-truth photo sites ──────────────────────────────────────────────────
# Edit photo_path for each site once images are available.
# Use None to render a placeholder tile instead.
PHOTO_SITES = {
    "Mohembo (inlet)":    {"lon": 21.803,  "lat": -18.281, "photo": None},
    "Seronga":            {"lon": 22.396,  "lat": -18.824, "photo": None},
    "Chief's Island":     {"lon": 23.118,  "lat": -19.410, "photo": None},
    "Maun (outlet)":      {"lon": 23.416,  "lat": -19.983, "photo": None},
}

# ── CRS constants ────────────────────────────────────────────────────────────
WGS84    = "EPSG:4326"
EA_CRS   = "EPSG:6933"   # WGS 84 / NSIDC EASE-Grid 2.0 Global (equal-area)

# ── Style ─────────────────────────────────────────────────────────────────────
DELTA_COLOR  = "#8c2d04"   # brick red
BBOX_COLOR   = "#fdae6b"   # warm orange
SITE_MARKER  = "^"
SITE_COLOR   = "gold"
SITE_EDGE    = "k"
DPI          = 150

print("Paths OK:", DELTA_SHP.exists(), LANDSAT_DSWE.exists(), SENTINEL_DSWE.exists())


## 2 · Load and Reproject the Delta Shapefile

In [ ]:
# ── Delta polygon ──────────────────────────────────────────────────────────────
delta_gdf = gpd.read_file(DELTA_SHP).to_crs(WGS84)
delta_union = delta_gdf.unary_union
delta_centroid = delta_union.centroid

# Tight bounding box (lon_min, lat_min, lon_max, lat_max)
minx, miny, maxx, maxy = delta_union.bounds
# Add a small margin for the map extent
MARGIN = 0.4   # degrees
map_extent = [minx - MARGIN, maxx + MARGIN, miny - MARGIN, maxy + MARGIN]
# cartopy order: [west, east, south, north]

# Bounding boxes — match the three geometry types used in ET-comparison notebook:
#   delta_polygon  → the exact delta shapefile polygon
#   poly_bbox      → tight bounding box derived from the delta polygon outline
#   okavango_bbox  → manually-specified wider study-area bbox
poly_bbox     = (minx, miny, maxx, maxy)              # tight bbox of delta polygon
okavango_bbox = (21.7913, -21.18, 25.0, -18.2694)     # manual wider study-area bbox

# Equal-area projection for area stats
delta_ea = gpd.GeoDataFrame(geometry=[delta_union], crs=WGS84).to_crs(EA_CRS)
delta_area_km2 = float(delta_ea.geometry.area.sum()) / 1e6

print(f"Delta centroid : {delta_centroid.y:.3f}°N, {delta_centroid.x:.3f}°E")
print(f"Bounding box   : {minx:.3f}–{maxx:.3f}°E, {miny:.3f}–{maxy:.3f}°N")
print(f"poly_bbox      : {poly_bbox}")
print(f"okavango_bbox  : {okavango_bbox}")
print(f"Delta area     : {delta_area_km2:,.0f} km²")

# ── Optional region layers ──────────────────────────────────────────────────
regions_gdf = gpd.read_file(REGIONS_PKG).to_crs(WGS84) if REGIONS_PKG.exists() else None
highlands_gdf = gpd.read_file(HIGHLANDS).to_crs(WGS84) if HIGHLANDS.exists() else None

# ── GRDC station metadata ───────────────────────────────────────────────────
if GRDC_META.exists():
    grdc_meta = pd.read_csv(GRDC_META)
    # Keep stations likely within the Okavango basin: rough lon/lat filter
    grdc_ok = grdc_meta[
        (grdc_meta["lon"].between(20, 26)) &
        (grdc_meta["lat"].between(-22, -17))
    ].copy()
else:
    grdc_ok = pd.DataFrame(columns=["station_name", "lon", "lat"])
    print("No GRDC metadata found — station layer will be empty")

# Always plot Mohembo as the primary gauge
MOHEMBO = {"name": "Mohembo\n(1357100)", "lon": 21.803, "lat": -18.281}
print(f"\nFound {len(grdc_ok)} GRDC stations in the map extent")


## 3 · Load DSWE Inundation Time Series

In [ ]:
# ── Load monthly DSWE CSVs ─────────────────────────────────────────────────────
def load_dswe(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"])
    df["sensor"] = label
    return df

ls  = load_dswe(LANDSAT_DSWE,  "Landsat")
s2  = load_dswe(SENTINEL_DSWE, "Sentinel-2")

dswe = pd.concat([ls, s2], ignore_index=True)
dswe["month"] = dswe["date"].dt.month

# ── Seasonal hydroperiod: mean km² by calendar month per sensor ────────────────
seasonal = (
    dswe.groupby(["sensor", "month"])["km2"]
    .mean()
    .reset_index()
    .rename(columns={"km2": "mean_km2"})
)

# Pivot for easy plotting
seasonal_pivot = seasonal.pivot(index="month", columns="sensor", values="mean_km2")

# ── Annual flood area (for reference) ─────────────────────────────────────────
annual = pd.read_csv(ANNUAL_FLOOD) if ANNUAL_FLOOD.exists() else None
if annual is not None:
    print("Annual flood CSV columns:", annual.columns.tolist())

MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

print("Seasonal mean inundation area (km²):")
print(seasonal_pivot.round(0).to_string())
print(f"\nLandsat:    {ls['date'].dt.year.min()}–{ls['date'].dt.year.max()}, n={len(ls)}")
print(f"Sentinel-2: {s2['date'].dt.year.min()}–{s2['date'].dt.year.max()}, n={len(s2)}")


## 4 · Build the Static Basemap (cartopy) with Overlays and Africa Locator Inset

In [ ]:
from shapely.geometry import box as _shp_box


def _add_scalebar(ax, length_km: float = 100, x0=0.05, y0=0.03,
                  transform=None, fontsize=7):
    """Draw a simple scale bar in axis-fraction coordinates."""
    if transform is None:
        transform = ax.transAxes
    # approximate degrees per km at the delta centroid latitude (~19°S)
    km_per_deg_lon = 111.32 * np.cos(np.radians(abs(delta_centroid.y)))
    deg_len = length_km / km_per_deg_lon
    ax_x0, ax_y0 = x0, y0
    ax.annotate(
        "",
        xy=(ax_x0 + 0.01 + deg_len / (map_extent[1] - map_extent[0]), ax_y0 + 0.01),
        xytext=(ax_x0 + 0.01, ax_y0 + 0.01),
        xycoords="axes fraction",
        arrowprops=dict(arrowstyle="-", color="k", lw=1.5),
    )
    ax.text(
        ax_x0 + 0.01 + deg_len / (map_extent[1] - map_extent[0]) / 2,
        ax_y0 + 0.025,
        f"{length_km} km",
        ha="center", va="bottom",
        fontsize=fontsize, transform=ax.transAxes,
    )


if HAS_CARTOPY:
    PROJ = ccrs.PlateCarree()

    # ─────────────────────────────────────────────
    # Main map axis
    # ─────────────────────────────────────────────
    fig_map, ax_main = plt.subplots(
        figsize=(8, 7),
        subplot_kw={"projection": PROJ},
    )

    west, east, south, north = map_extent
    ax_main.set_extent([west, east, south, north], crs=PROJ)

    # Natural Earth features
    SCALE = "10m"
    ax_main.add_feature(cfeature.NaturalEarthFeature(
        "physical", "land", SCALE,
        facecolor=cfeature.COLORS["land"], edgecolor="none"))
    ax_main.add_feature(cfeature.NaturalEarthFeature(
        "physical", "ocean", SCALE,
        facecolor=cfeature.COLORS["water"], edgecolor="none"))
    ax_main.add_feature(cfeature.NaturalEarthFeature(
        "cultural", "admin_0_countries", SCALE,
        facecolor="none", edgecolor="#888", linewidth=0.5))
    ax_main.add_feature(cfeature.NaturalEarthFeature(
        "physical", "rivers_lake_centerlines", SCALE,
        facecolor="none", edgecolor="#5bafd6", linewidth=0.7))
    ax_main.add_feature(cfeature.NaturalEarthFeature(
        "physical", "lakes", SCALE,
        facecolor=cfeature.COLORS["water"], edgecolor="#5bafd6", linewidth=0.3))

    # Delta polygon overlay
    ax_main.add_geometries(
        [delta_union], crs=PROJ,
        facecolor=DELTA_COLOR + "33",   # 20% transparent
        edgecolor=DELTA_COLOR,
        linewidth=1.5,
        label="Okavango Delta",
        zorder=4,
    )

    # poly_bbox — tight bounding box of the delta polygon (orange dashed)
    ax_main.add_geometries(
        [_shp_box(*poly_bbox)], crs=PROJ,
        facecolor="none", edgecolor="#fdae6b",
        linewidth=1.2, linestyle="--", zorder=3,
    )

    # okavango_bbox — manual wider study-area bbox (dark-orange dotted)
    ax_main.add_geometries(
        [_shp_box(*okavango_bbox)], crs=PROJ,
        facecolor="none", edgecolor="#d95f02",
        linewidth=1.0, linestyle=":", zorder=3,
    )

    # Region sub-polygons (if available)
    if regions_gdf is not None:
        ax_main.add_geometries(
            regions_gdf.geometry, crs=PROJ,
            facecolor="none", edgecolor="#888",
            linewidth=0.8, linestyle="--", zorder=3,
        )

    # GRDC stations
    if len(grdc_ok) > 0:
        ax_main.scatter(
            grdc_ok["lon"], grdc_ok["lat"],
            marker="o", c="#1a9850", s=20, zorder=6,
            transform=PROJ, label="GRDC stations",
        )

    # Mohembo gauge (primary upstream boundary)
    ax_main.scatter(
        MOHEMBO["lon"], MOHEMBO["lat"],
        marker=SITE_MARKER, c="gold", s=80, edgecolors="k", linewidths=0.6,
        zorder=7, transform=PROJ, label="Mohembo gauge",
    )
    ax_main.text(
        MOHEMBO["lon"] + 0.08, MOHEMBO["lat"] - 0.1,
        "Mohembo", fontsize=7, transform=PROJ,
        verticalalignment="top", color="k",
    )

    # Photo site markers
    for site_name, info in PHOTO_SITES.items():
        ax_main.scatter(
            info["lon"], info["lat"],
            marker="*", c=SITE_COLOR, s=60, edgecolors=SITE_EDGE, linewidths=0.5,
            zorder=8, transform=PROJ,
        )

    # Gridlines
    gl = ax_main.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle=":",
        x_inline=False, y_inline=False,
    )
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}

    # Scale bar
    _add_scalebar(ax_main, length_km=100, fontsize=7)

    # Legend
    legend_handles = [
        mpatches.Patch(facecolor=DELTA_COLOR + "33", edgecolor=DELTA_COLOR,
                       linewidth=1.5, label="Okavango Delta"),
        plt.Line2D([0], [0], color="#fdae6b", lw=1.2, linestyle="--",
                   label="Polygon tight bbox"),
        plt.Line2D([0], [0], color="#d95f02", lw=1.0, linestyle=":",
                   label="Okavango bbox"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#1a9850",
                   markersize=5, label="GRDC stations"),
        plt.Line2D([0], [0], marker=SITE_MARKER, color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=7, label="Mohembo gauge"),
    ]
    ax_main.legend(handles=legend_handles, loc="lower right", fontsize=7,
                   framealpha=0.9)

    ax_main.set_title("Okavango Delta — Study Region", fontsize=10, pad=6)

    # ─────────────────────────────────────────────────────
    # Africa locator inset
    # ─────────────────────────────────────────────────────
    ax_africa = fig_map.add_axes([0.04, 0.62, 0.22, 0.30],
                                  projection=PROJ)
    ax_africa.set_extent([-20, 55, -38, 38], crs=PROJ)
    ax_africa.add_feature(cfeature.NaturalEarthFeature(
        "physical", "land", "110m",
        facecolor="#d9d9d9", edgecolor="none"))
    ax_africa.add_feature(cfeature.NaturalEarthFeature(
        "physical", "ocean", "110m",
        facecolor="#c6e2f5", edgecolor="none"))
    ax_africa.add_feature(cfeature.NaturalEarthFeature(
        "cultural", "admin_0_countries", "110m",
        facecolor="none", edgecolor="#999", linewidth=0.3))
    # Red star on the delta centroid
    ax_africa.scatter(
        delta_centroid.x, delta_centroid.y,
        marker="*", c="red", s=40, zorder=5, transform=PROJ,
    )
    ax_africa.set_title("Africa", fontsize=6, pad=2)
    for spine in ax_africa.spines.values():
        spine.set_edgecolor("#333")
        spine.set_linewidth(0.8)

    plt.tight_layout(pad=1.5)
    plt.show()
    print("Main map built.")

else:
    # ── Fallback: geopandas + contextily ──────────────────────────────────────────
    fig_map, ax_main = plt.subplots(figsize=(8, 7))
    delta_gdf.plot(ax=ax_main, facecolor=DELTA_COLOR + "33",
                   edgecolor=DELTA_COLOR, linewidth=1.5)
    if HAS_CONTEXTILY:
        delta_gdf_web = delta_gdf.to_crs("EPSG:3857")
        fig2, ax2 = plt.subplots(figsize=(8, 7))
        delta_gdf_web.plot(ax=ax2, facecolor=DELTA_COLOR + "33",
                           edgecolor=DELTA_COLOR, linewidth=1.5)
        cx.add_basemap(ax2, source=cx.providers.Esri.WorldImagery, zoom=7)
        ax2.set_axis_off()
        ax2.set_title("Okavango Delta — Study Region (contextily)", fontsize=10)
        plt.tight_layout()
        plt.show()
        ax_main = ax2
    else:
        ax_main.set_xlim(minx - MARGIN, maxx + MARGIN)
        ax_main.set_ylim(miny - MARGIN, maxy + MARGIN)
        ax_main.set_aspect("equal")
        ax_main.set_xlabel("Longitude (°E)")
        ax_main.set_ylabel("Latitude (°N)")
        ax_main.set_title("Okavango Delta — Study Region", fontsize=10)
        plt.tight_layout()
        plt.show()
    print("Fallback map built (cartopy not available).")


## 5 · DSWE Hydroperiod Seasonal Stack (Inset Panel)

In [ ]:
# ── Colour palette ─────────────────────────────────────────────────────────────
SENSOR_COLORS = {
    "Landsat":    "#2166ac",   # blue
    "Sentinel-2": "#d6604d",   # salmon-red
}

# ── Bar chart: monthly mean inundated area ─────────────────────────────────────
fig_hydro, ax_hydro = plt.subplots(figsize=(7, 3.5))

x   = np.arange(1, 13)
w   = 0.35
sensors = seasonal_pivot.columns.tolist()
offsets = np.linspace(-w / 2 * (len(sensors) - 1), w / 2 * (len(sensors) - 1), len(sensors))

for sensor, offset in zip(sensors, offsets):
    if sensor not in seasonal_pivot.columns:
        continue
    vals = seasonal_pivot[sensor].reindex(x).values
    ax_hydro.bar(
        x + offset, vals, width=w,
        color=SENSOR_COLORS.get(sensor, "gray"),
        alpha=0.85, label=sensor,
        edgecolor="white", linewidth=0.3,
    )

ax_hydro.set_xticks(x)
ax_hydro.set_xticklabels(MONTH_LABELS, fontsize=8)
ax_hydro.set_ylabel("Mean inundated area (km²)", fontsize=9)
ax_hydro.set_xlabel("Month", fontsize=9)
ax_hydro.set_title("Okavango Delta DSWE hydroperiod\n(monthly mean inundation area)", fontsize=9)
ax_hydro.legend(fontsize=8, framealpha=0.9)
ax_hydro.spines["top"].set_visible(False)
ax_hydro.spines["right"].set_visible(False)
ax_hydro.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f"{v:,.0f}"))

fig_hydro.tight_layout()
fig_hydro.savefig(FIG_OUT / "fig_study_area_hydroperiod.png", dpi=DPI, bbox_inches="tight")
plt.show()
print("Hydroperiod panel saved.")


## 6 · Ground-Truth Photo Insets at Key Locations

In [ ]:
def _placeholder_photo(ax, label: str, color: str = "#b2d8d8") -> None:
    """Draw a labelled placeholder tile when no photo is available."""
    ax.set_facecolor(color)
    ax.text(
        0.5, 0.5, label.replace(" ", "\n"),
        ha="center", va="center", transform=ax.transAxes,
        fontsize=6, style="italic", color="#444",
        wrap=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor("#666")
        spine.set_linewidth(0.8)


def _load_photo(path: Path, ax) -> None:
    """Display an image file in an Axes."""
    img = mpimg.imread(str(path))
    ax.imshow(img, aspect="auto")
    ax.set_xticks([])
    ax.set_yticks([])


# ── 2×2 Photo grid ────────────────────────────────────────────────────────────
SITES = list(PHOTO_SITES.items())
n_photos = len(SITES)   # 4

fig_photos, axes_ph = plt.subplots(
    2, 2, figsize=(5, 5),
    gridspec_kw={"hspace": 0.35, "wspace": 0.2},
)
fig_photos.suptitle("Ground-truth sites", fontsize=9, y=1.01)

for ax_ph, (site_name, info) in zip(axes_ph.ravel(), SITES):
    photo_path = info.get("photo")
    if photo_path is not None and Path(photo_path).exists():
        _load_photo(Path(photo_path), ax_ph)
    else:
        _placeholder_photo(ax_ph, site_name + "\n(photo TBD)")
    ax_ph.set_title(site_name, fontsize=7, pad=2)
    ax_ph.text(
        0.02, 0.02,
        f"{info['lat']:.2f}°N, {info['lon']:.2f}°E",
        transform=ax_ph.transAxes, fontsize=5, color="#333",
        va="bottom",
    )

fig_photos.tight_layout()
fig_photos.savefig(FIG_OUT / "fig_study_area_photos.png", dpi=DPI, bbox_inches="tight")
plt.show()
print("Photo panel saved.")


## 7 · Compose and Export the Full `fig_study_area` Figure

Assemble all panels into a single GridSpec figure:

```
┌───────────────────┬────────────────────┐
│                   │  (b) Hydroperiod   │
│   (a) Main map    │  seasonal stack    │
│                   ├────────────────────┤
│                   │  (c)  Photos 2×2   │
└───────────────────┴────────────────────┘
```

In [ ]:
from shapely.geometry import box as _shp_box

FIG_W, FIG_H = 14, 9   # inches (landscape)

if HAS_CARTOPY:
    # ── GridSpec layout ──────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(FIG_W, FIG_H), constrained_layout=False)

    gs = GridSpec(
        nrows=2, ncols=2,
        figure=fig,
        width_ratios=[1.55, 1],
        height_ratios=[1, 1],
        left=0.04, right=0.97,
        bottom=0.06, top=0.93,
        hspace=0.35, wspace=0.12,
    )

    # ── (a) Main map — spans both rows on the left ────────────────────────────
    ax_a = fig.add_subplot(gs[:, 0], projection=ccrs.PlateCarree())
    ax_a.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

    ax_a.add_feature(cfeature.NaturalEarthFeature(
        "physical", "land", "10m",
        facecolor=cfeature.COLORS["land"], edgecolor="none"))
    ax_a.add_feature(cfeature.NaturalEarthFeature(
        "physical", "ocean", "10m",
        facecolor=cfeature.COLORS["water"], edgecolor="none"))
    ax_a.add_feature(cfeature.NaturalEarthFeature(
        "cultural", "admin_0_countries", "10m",
        facecolor="none", edgecolor="#888", linewidth=0.45))
    ax_a.add_feature(cfeature.NaturalEarthFeature(
        "physical", "rivers_lake_centerlines", "10m",
        facecolor="none", edgecolor="#5bafd6", linewidth=0.6))
    ax_a.add_feature(cfeature.NaturalEarthFeature(
        "physical", "lakes", "10m",
        facecolor=cfeature.COLORS["water"], edgecolor="#5bafd6", linewidth=0.3))

    # Delta polygon
    ax_a.add_geometries(
        [delta_union], crs=ccrs.PlateCarree(),
        facecolor=DELTA_COLOR + "33", edgecolor=DELTA_COLOR, linewidth=1.5, zorder=4)

    # poly_bbox — tight bounding box of the delta polygon (orange dashed)
    ax_a.add_geometries(
        [_shp_box(*poly_bbox)], crs=ccrs.PlateCarree(),
        facecolor="none", edgecolor="#fdae6b",
        linewidth=1.1, linestyle="--", zorder=3)

    # okavango_bbox — manual wider study-area bbox (dark-orange dotted)
    ax_a.add_geometries(
        [_shp_box(*okavango_bbox)], crs=ccrs.PlateCarree(),
        facecolor="none", edgecolor="#d95f02",
        linewidth=0.9, linestyle=":", zorder=3)

    if regions_gdf is not None:
        ax_a.add_geometries(
            regions_gdf.geometry, crs=ccrs.PlateCarree(),
            facecolor="none", edgecolor="#888",
            linewidth=0.7, linestyle="--", zorder=3)

    if len(grdc_ok) > 0:
        ax_a.scatter(grdc_ok["lon"], grdc_ok["lat"],
                     marker="o", c="#1a9850", s=18, zorder=6,
                     transform=ccrs.PlateCarree(), label="GRDC stations")

    ax_a.scatter(MOHEMBO["lon"], MOHEMBO["lat"],
                 marker=SITE_MARKER, c="gold", s=90, edgecolors="k", linewidths=0.7,
                 zorder=7, transform=ccrs.PlateCarree(), label="Mohembo gauge")
    ax_a.text(MOHEMBO["lon"] + 0.08, MOHEMBO["lat"] - 0.08, "Mohembo",
              fontsize=6.5, transform=ccrs.PlateCarree(), va="top")

    for site_name, info in PHOTO_SITES.items():
        ax_a.scatter(info["lon"], info["lat"],
                     marker="*", c=SITE_COLOR, s=55, edgecolors=SITE_EDGE, linewidths=0.5,
                     zorder=8, transform=ccrs.PlateCarree())

    gl_a = ax_a.gridlines(draw_labels=True, linewidth=0.3, color="gray",
                           alpha=0.6, linestyle=":", x_inline=False, y_inline=False)
    gl_a.top_labels = False
    gl_a.right_labels = False
    gl_a.xlabel_style = {"size": 6}
    gl_a.ylabel_style = {"size": 6}

    _add_scalebar(ax_a, length_km=100, fontsize=6.5)

    _handles = [
        mpatches.Patch(facecolor=DELTA_COLOR + "33", edgecolor=DELTA_COLOR,
                       linewidth=1.5, label="Okavango Delta"),
        plt.Line2D([0], [0], color="#fdae6b", lw=1.1, linestyle="--",
                   label="Polygon tight bbox"),
        plt.Line2D([0], [0], color="#d95f02", lw=0.9, linestyle=":",
                   label="Okavango bbox"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#1a9850",
                   markersize=5, label="GRDC stations"),
        plt.Line2D([0], [0], marker=SITE_MARKER, color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=7, label="Mohembo gauge"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor=SITE_COLOR,
                   markeredgecolor="k", markersize=7, label="Field sites"),
    ]
    ax_a.legend(handles=_handles, loc="lower right", fontsize=6.5, framealpha=0.9)
    ax_a.set_title("(a)  Study region", fontsize=9, pad=5, loc="left")

    # Africa locator inset on panel (a)
    ax_africa_inset = fig.add_axes([0.04, 0.60, 0.14, 0.20],
                                    projection=ccrs.PlateCarree())
    ax_africa_inset.set_extent([-20, 55, -38, 38], crs=ccrs.PlateCarree())
    ax_africa_inset.add_feature(cfeature.NaturalEarthFeature(
        "physical", "land", "110m", facecolor="#d9d9d9", edgecolor="none"))
    ax_africa_inset.add_feature(cfeature.NaturalEarthFeature(
        "physical", "ocean", "110m", facecolor="#c6e2f5", edgecolor="none"))
    ax_africa_inset.add_feature(cfeature.NaturalEarthFeature(
        "cultural", "admin_0_countries", "110m",
        facecolor="none", edgecolor="#aaa", linewidth=0.25))
    ax_africa_inset.scatter(delta_centroid.x, delta_centroid.y,
                             marker="*", c="red", s=40, zorder=5,
                             transform=ccrs.PlateCarree())
    ax_africa_inset.set_title("Africa", fontsize=5.5, pad=1)
    for sp in ax_africa_inset.spines.values():
        sp.set_edgecolor("#444")
        sp.set_linewidth(0.8)

    # ── (b) Hydroperiod bar chart (top-right) ──────────────────────────────────
    ax_b = fig.add_subplot(gs[0, 1])
    x   = np.arange(1, 13)
    w   = 0.35
    sensors = seasonal_pivot.columns.tolist()
    offsets = np.linspace(-w / 2 * (len(sensors) - 1), w / 2 * (len(sensors) - 1), len(sensors))
    for sensor, offset in zip(sensors, offsets):
        if sensor not in seasonal_pivot.columns:
            continue
        vals = seasonal_pivot[sensor].reindex(x).values
        ax_b.bar(x + offset, vals, width=w,
                 color=SENSOR_COLORS.get(sensor, "gray"),
                 alpha=0.85, label=sensor,
                 edgecolor="white", linewidth=0.3)
    ax_b.set_xticks(x)
    ax_b.set_xticklabels(MONTH_LABELS, fontsize=6.5, rotation=45, ha="right")
    ax_b.set_ylabel("Mean inundated\narea (km²)", fontsize=7)
    ax_b.legend(fontsize=6.5, framealpha=0.9, loc="upper left")
    ax_b.spines["top"].set_visible(False)
    ax_b.spines["right"].set_visible(False)
    ax_b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax_b.tick_params(axis="both", labelsize=6.5)
    ax_b.set_title("(b)  DSWE hydroperiod\n(monthly mean, all years)", fontsize=8, pad=4, loc="left")

    # ── (c) Photo insets 2×2 (bottom-right) ───────────────────────────────────
    gs_inner = gs[1, 1].subgridspec(2, 2, hspace=0.45, wspace=0.2)
    photo_axes = [fig.add_subplot(gs_inner[r, c]) for r in range(2) for c in range(2)]

    for ax_ph, (site_name, info) in zip(photo_axes, SITES):
        photo_path = info.get("photo")
        if photo_path is not None and Path(photo_path).exists():
            _load_photo(Path(photo_path), ax_ph)
        else:
            _placeholder_photo(ax_ph, site_name)
        ax_ph.set_title(site_name.split(" (")[0], fontsize=5.5, pad=2)
        ax_ph.text(0.02, 0.03, f"{info['lat']:.2f}°, {info['lon']:.2f}°",
                   transform=ax_ph.transAxes, fontsize=4.5, color="#333", va="bottom")

    photo_axes[0].set_title(
        "(c)  Ground-truth sites", fontsize=8, pad=14, loc="left",
        fontdict={"fontsize": 8},
    )

    # ── Overall title ──────────────────────────────────────────────────────────
    fig.suptitle(
        "Okavango Delta — Study Region, DSWE Hydroperiod, and Field Sites",
        fontsize=11, y=0.97,
    )

    fig.savefig(FIG_OUT / "fig_study_area.png",  dpi=300, bbox_inches="tight")
    fig.savefig(FIG_OUT / "fig_study_area.pdf",  bbox_inches="tight")
    plt.show()
    print(f"Saved → {(FIG_OUT / 'fig_study_area.png').resolve()}")

else:
    print("cartopy not available — assemble panels manually from the individual files saved above.")
    print(f"  Hydroperiod: {FIG_OUT / 'fig_study_area_hydroperiod.png'}")
    print(f"  Photos:      {FIG_OUT / 'fig_study_area_photos.png'}")


## 8 · Update Figure Registry

In [ ]:
import datetime

REGISTRY = ROOT / "figures" / "registry.txt"

entry = (
    f"fig_study_area | "
    f"figures/study_area/fig_study_area.png | "
    f"figures/study_area/fig_study_area.pdf | "
    f"generated {datetime.date.today().isoformat()} | "
    f"notebook: notebooks/fig_study_area.ipynb\n"
)

existing = REGISTRY.read_text() if REGISTRY.exists() else ""

# Overwrite/update the entry for this figure
lines = [ln for ln in existing.splitlines(keepends=True)
         if not ln.startswith("fig_study_area")]
lines.append(entry)
REGISTRY.write_text("".join(lines))

print("Registry updated:")
print(entry.strip())


---
## Notes and Customisation

### Adding real ground-truth photos
Edit `PHOTO_SITES` in cell 1 to give each site a `"photo"` path, e.g.:
```python
PHOTO_SITES = {
    "Mohembo (inlet)": {"lon": 21.803, "lat": -18.281,
                        "photo": "../assets/photos/mohembo_360.jpg"},
    ...
}
```
Photos can be standard JPEG/PNG; 360° equirectangular images are treated the same way — the cell just calls `mpimg.imread`.

### Adding a satellite basemap tile
If `contextily` is installed, uncomment the `cx.add_basemap(...)` block inside the cartopy `else` branch, or merge it into the cartopy path by converting the delta GeoDataFrame to EPSG:3857 and using `ax.imshow` with a WMS URL.

### Adjusting the map extent / zoom
Change `MARGIN` in cell 2 or edit `map_extent` directly.

### Figure DPI and dimensions
`DPI = 300` and `FIG_W, FIG_H = 14, 9` are set in cell 1 and cell 7 respectively.

### EE / DSWE raster composites
The current seasonal inset uses the pre-computed monthly CSV (`monthly_landsat_dswe.csv`).  
To replace it with actual spatially-resolved EE raster composites:
1. Run Earth Engine to export seasonal mean DSWE images.
2. Load them as GeoTIFFs via `rioxarray` or `rasterio` and replace the bar chart in panel (b) with four `ax.imshow` calls arranged in a 1×4 grid.

Last updated: 2026